# Build a vector database (embeddings DB) with Gemini

Walks through the mechanics behind `samples/vector_store.py` step by step: chunk documents → embed each chunk with Gemini → persist vectors to disk → query by cosine similarity. This is the retrieval leg that powers the RAG search agent in `10_rag_search_agent.ipynb`.

**Setup** — assumes `.env` is configured at the project root and you've run `gcloud auth application-default login`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join('..', 'samples')))

import glob
import numpy as np
from vector_store import Chunk, chunk_text, embed_texts, INDEX_PATH


## Step 1 — chunk the source documents

`data/docs/*.txt` holds a few internal policy docs. `chunk_text()` splits each one on blank lines (paragraphs), then further splits any paragraph longer than `max_chars` — small chunks make for more precise retrieval than embedding a whole document at once.

In [ ]:
docs_glob = os.path.join('..', 'data', 'docs', '*.txt')
chunks: list[Chunk] = []
for path in sorted(glob.glob(docs_glob)):
    with open(path, 'r', encoding='utf-8') as f:
        chunks.extend(chunk_text(f.read(), source=os.path.basename(path)))

print(f'{len(chunks)} chunks from {len(set(c.source for c in chunks))} documents')
for c in chunks:
    print(f'  [{c.source}] {c.text[:70]!r}')


## Step 2 — embed each chunk with Gemini

`embed_texts()` calls Vertex AI's `text-embedding-005` model and returns one 768-dimensional vector per chunk. `task_type="RETRIEVAL_DOCUMENT"` tells the model these are documents being indexed (as opposed to a search query — Gemini's embedding model optimizes the vector space differently for each).

In [ ]:
vectors = embed_texts([c.text for c in chunks], task_type='RETRIEVAL_DOCUMENT')
print('vectors shape:', vectors.shape)
print('first vector, first 8 dims:', vectors[0][:8])


## Step 3 — persist the index to disk

Save the vectors alongside their source text and filename into a single `.npz` file — a minimal embeddings database. At this scale (a handful of docs) plain numpy is plenty; swap this cell for Vertex AI Vector Search, pgvector, or Chroma if you need this to scale to millions of chunks.

In [ ]:
os.makedirs(os.path.dirname(INDEX_PATH), exist_ok=True)
np.savez(
    INDEX_PATH,
    vectors=vectors,
    texts=np.array([c.text for c in chunks], dtype=object),
    sources=np.array([c.source for c in chunks], dtype=object),
)
print('wrote', INDEX_PATH)


## Inspect the resulting database

(Equivalently, you could have just called `vector_store.build_index()` to do steps 1–3 in one call — it's the same logic.)

In [ ]:
data = np.load(INDEX_PATH, allow_pickle=True)
print('arrays:', list(data.keys()))
for k in data.keys():
    print(f'  {k}: shape={data[k].shape}, dtype={data[k].dtype}')


## Step 4 — query the database

Embed the query with `task_type="RETRIEVAL_QUERY"` (the query-side counterpart to `RETRIEVAL_DOCUMENT` above), then rank every stored chunk by cosine similarity.

In [ ]:
from vector_store import search

for text, source, score in search('how much PTO carries over?', top_k=3):
    print(f'[{score:.3f}] ({source}) {text}')
